# Run-by-run sensor stability on synthetic data

This notebook tests `stdiag.run_stability` on a synthetic series of 60 runs. It applies three
change-point methods (CUSUM, PELT and Bayesian online change-point detection) to the per-run residual
mean and width of every MVTX chip and INTT sensor, and compares the result with the injected truth.

Everything runs locally from the fit tables. It needs no ROOT and no AI. Outputs go to `notebooks/output/` (git-ignored).

**Contents**
1. Why residuals versus run, and what they can and cannot tell you
2. Make the synthetic run series (and a series with no changes)
3. Load the tables; look at the data
4. Common mode: layer-wide changes versus sensor changes
5. CUSUM, 6. PELT, 7. BOCPD, each on one sensor
8. Thresholds and false alarms (series with no changes)
9. Full scan versus the injected truth
10. Detection efficiency versus step size
11. Drifts, outlier runs and width changes
12. Running it on real sPHENIX data

## 1. Why residuals versus run

For every cluster on a track, `ntp` stores the local residual (cluster minus track) in r-φ and z.
`macros/plot_fits_overlaid_stavechip.C` fits, per input, the residual distribution of every chip/sensor
and writes μ (mean), σ (width) and their errors to `residual_fits_<label>.csv`.
With **one input per run**, μ(run) for one sensor is a time series. If the sensor moves by δ relative to the
geometry used in reconstruction, μ moves by about δ (with a sign set by the residual convention).

What a change in μ(run) does **not** prove:
- **Same conditions are required.** It means "sensor moved" only if every run was reconstructed with the same alignment/geometry tag,
  field, track selection and (if the TPC is in the fit) the same distortion corrections. A new alignment tag
  changes μ for every sensor, which is why the common mode is scanned separately.
- **Biased residuals.** The sensor is part of the track fit, so the residual shift is smaller than the true
  displacement. The size of that pull depends on the fit; the synthetic data does not model it.
- **Weak modes.** Radial expansion, twist, telescope and rigid movements of the whole tracker barely change residuals.
- **Run-to-run systematics.** Beam position, luminosity, track momentum spectrum and pile-up can change from run to run.
  The scan adds a robust per-series "extra scatter" term on top of the fit errors, but it can't remove a real systematic.

To compare **positions** across runs reconstructed with different tags: position ≈ nominal + alignment
parameters of that tag + measured residual shift (with the sign convention of `residualX/Y` in MakeMilleFiles).

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
OUT = ROOT / "notebooks" / "output"
OUT.mkdir(parents=True, exist_ok=True)

from stdiag import run_stability as rs
from synthetic import make_synthetic_run_series as gen
from synthetic import check_run_stability as truth

UM = 1e-3            # mm per micrometre
# Reference palette (light mode) from the data-viz method: three categorical
# slots validated all-pairs; aqua is below 3:1 on the surface, so every
# method line also has its own marker and a direct label.
SURFACE, INK, INK2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#a3a29c"
METHOD_COLOR = {"cusum": "#2a78d6", "pelt": "#eb6834", "bocpd": "#1baf7a"}
METHOD_MARKER = {"cusum": "o", "pelt": "s", "bocpd": "^"}
DIVERGING = LinearSegmentedColormap.from_list(
    "blue_gray_red", ["#184f95", "#3987e5", "#f0efec", "#e34948", "#9e2b2b"])
SEQUENTIAL = LinearSegmentedColormap.from_list(
    "blue", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK2, "text.color": INK,
    "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True,
    "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "axes.spines.top": False,
    "axes.spines.right": False, "lines.linewidth": 2, "font.size": 10,
    "legend.frameon": False, "figure.dpi": 110})
pd.set_option("display.width", 200)

## 2. Make the synthetic run series

`synthetic/make_synthetic_run_series.py` writes one `residual_fits_run<N>.csv` per run, with the full MVTX/INTT geometry
(1312 rows per run). The model per sensor and coordinate:
- a static offset and width (as in the existing two-input generator);
- per-run jitter of the mean: 1 µm for MVTX, 4 µm for INTT r-φ, 0.3 mm for INTT z;
- a per-run layer jitter (half of that, common to every sensor of a layer);
- 2% width jitter;
- a per-run luminosity factor on the entries (0.3–1.0).

The jitter sizes are a guess, not a measurement. The second series (`--null`) has the same random numbers but no
position or width changes; only the luminosity and missing-ladder effects remain.

In [ ]:
def write_series(folder: Path, null: bool) -> list[Path]:
    tables, run_info = gen.build_run_series(gen.DEFAULT_SEED, gen.DEFAULT_RUNS,
                                            gen.active_injections(null))
    folder.mkdir(parents=True, exist_ok=True)
    for old in folder.glob("residual_fits_run*.csv"):
        old.unlink()
    for label, table in tables.items():
        table.to_csv(folder / f"residual_fits_{label}.csv", index=False,
                     float_format="%.8g")
    run_info.to_csv(folder / "run_info.csv", index=False)
    return sorted(folder.glob("residual_fits_run*.csv"))

series_paths = write_series(OUT / "series", null=False)
null_paths = write_series(OUT / "null", null=True)
print(len(series_paths), "run tables;", len(null_paths), "null run tables")
gen.injection_table(gen.INJECTIONS)[["name", "kind", "detector", "layer", "staves", "chips",
                                     "coord", "start", "end", "offset_mm", "sigma_scale",
                                     "expected"]]

## 3. Load the tables and look at the data

`load_run_series` orders the runs by the number in each label (or by `--order FILE`). `build_matrices` makes
one row per sensor and coordinate and one column per run. Runs whose fit status is not `ok`, and missing rows, become gaps (NaN), not zeros.

In [ ]:
long, runs = rs.load_run_series(series_paths)
run_info = pd.read_csv(OUT / "series" / "run_info.csv")
keys, M = rs.build_matrices(long, len(runs))
print(f"{len(runs)} runs ({runs['run'].iloc[0]} .. {runs['run'].iloc[-1]}), "
      f"{len(keys)} sensor/coordinate series")
runs.head()

**μ versus run for every MVTX chip.** Each row is one chip (stave × 9 + chip), with its mean over the first 5 runs
subtracted. The colour scale is diverging: blue means negative, gray means no change, red means positive. Visible: the L1 stave-4
step (run 30), the L2 half-layer z step (run 40, staves 0–9), the L1 layer-wide shift (run 52) and the bad run 47
(scattered colours in every row).

In [ ]:
def baseline_subtracted(X):
    return X - np.nanmedian(X[:, :5], axis=1, keepdims=True)

fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex=True, constrained_layout=True)
for column, layer in enumerate((0, 1, 2)):
    for row, coord in enumerate(("rphi", "z")):
        ax = axes[row, column]
        rows = np.flatnonzero((keys["detector"] == "MVTX") & (keys["layer"] == layer)
                              & (keys["coord"] == coord))
        image = ax.imshow(baseline_subtracted(M["mu"][rows]) / UM, aspect="auto",
                          cmap=DIVERGING, vmin=-10, vmax=10, interpolation="nearest")
        ax.set_title(f"MVTX L{layer} {coord}", loc="left", fontsize=10)
        ax.grid(False)
        if column == 0:
            ax.set_ylabel("stave × 9 + chip")
        if row == 1:
            ax.set_xlabel("run index")
fig.colorbar(image, ax=axes, shrink=0.8, label="μ − baseline [µm]")
fig.suptitle("Residual mean versus run, MVTX", x=0.01, ha="left")
plt.show()

**Single sensors.** Shown with fit errors; the dashed line marks the injected run. The y axes are in µm.

In [ ]:
def series_of(detector, layer, stave, chip, coord, X=None):
    row = keys.index[(keys["detector"] == detector) & (keys["layer"] == layer)
                     & (keys["stave"] == stave) & (keys["chip"] == chip)
                     & (keys["coord"] == coord)][0]
    return row, (M["mu"] if X is None else X)[row]

examples = {"step_chip": ("MVTX", 0, 5, 3, "rphi"), "transient": ("MVTX", 0, 8, 2, "rphi"),
            "drift": ("MVTX", 1, 10, 4, "rphi"), "outlier": ("MVTX", 0, 1, 1, "rphi")}
injected = {inj["name"]: inj for inj in gen.INJECTIONS}
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True, constrained_layout=True)
for ax, (name, key) in zip(axes.ravel(), examples.items()):
    row, x = series_of(*key)
    ax.errorbar(np.arange(len(x)), (x - np.nanmedian(x[:5])) / UM, yerr=M["mu_err"][row] / UM,
                fmt="o", ms=4, color=METHOD_COLOR["cusum"], ecolor=MUTED, elinewidth=1)
    ax.axvline(injected[name]["start"], color=INK2, ls="--", lw=1)
    ax.set_title(f"{name}: {' '.join(map(str, key))}", loc="left", fontsize=10)
    ax.set_ylabel("μ − baseline [µm]")
for ax in axes[1]:
    ax.set_xlabel("run index")
plt.show()

## 4. Common mode

The run-by-run **common mode** of a layer and coordinate is the clipped mean of μ over all its sensors (`--common-mode offset`).
`--common-mode harmonic` fits a + b sin φ + c cos φ over stave azimuth instead (layer rotation plus x/y translation).
The scan
- looks for change points in the common-mode series itself (`layer_mode`);
- scans every sensor **relative** to its layer (common mode subtracted) and **raw**;
- drops raw changes that the relative frame already has, or that a reported layer-mode change explains.

The case to watch is a **partial-layer movement**. Moving half of L2 by +6 µm moves the clipped mean by about +3 µm. That is below
the 5 µm minimum, so no layer-mode change is reported, but after subtraction the unmoved half looks moved by −3 µm.
So a relative-frame change is kept only if the sensor's raw series also changed by more than the minimum shift,
unless a layer-mode change was reported there. The raw frame then reports the moved staves, and they are grouped
into `coherent_stave` and `multi_stave` events.

In [ ]:
result = rs.scan(long, runs, rs.Config())
rs.annotate(result["events"], run_info, runs, 2)
cm = result["common_mode"]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True, constrained_layout=True)
for ax, (layer, coord, start) in zip(axes, ((1, "rphi", 52), (2, "z", 40))):
    index = cm["keys"].index[(cm["keys"]["detector"] == "MVTX") & (cm["keys"]["layer"] == layer)
                             & (cm["keys"]["coord"] == coord)][0]
    values = cm["values"][index]
    ax.errorbar(np.arange(len(values)), (values - np.nanmedian(values[:5])) / UM,
                yerr=cm["errors"][index] / UM, fmt="o", ms=4, color=METHOD_COLOR["cusum"],
                ecolor=MUTED, elinewidth=1)
    ax.axvline(start, color=INK2, ls="--", lw=1)
    ax.axhline(5, color=MUTED, lw=1); ax.axhline(-5, color=MUTED, lw=1)
    ax.text(1, 5.3, "minimum shift", color=INK2, fontsize=8)
    ax.set_title(f"Common mode MVTX L{layer} {coord}", loc="left", fontsize=10)
    ax.set_xlabel("run index")
axes[0].set_ylabel("common mode − baseline [µm]")
plt.show()
print("Whole-layer shift at run 52: layer_mode found;",
      "half-layer shift at run 40: about +3 µm, below the minimum shift.")
print("Relative-frame changes dropped as induced by the common mode:",
      result["events"].attrs["common_mode_induced_dropped"])

In [ ]:
counts = {}
for mode in ("offset", "harmonic", "none"):
    scanned = result if mode == "offset" else rs.scan(long, runs, rs.Config(common_mode=mode))
    counts[mode] = scanned["events"]["class"].value_counts()
pd.DataFrame(counts).fillna(0).astype(int)

With `none`, the layer-1 shift at run 52 is not recognised as one layer-wide change: it appears as a flood of
sensor steps grouped into 16 coherent staves. The drifting chip in that layer also carries the extra 7 µm step and is
no longer classified as a pure drift. `offset` and `harmonic` both recover the injected truth here.

## 5. CUSUM

The two-sided tabular CUSUM (Page 1954) works in units of each run's total error σ_tot:
$S^+_t = \max(0, S^+_{t-1} + z_t - k)$ and $S^-_t = \max(0, S^-_{t-1} - z_t - k)$, with $z_t = (x_t - \mu_0)/\sigma_{tot,t}$.
The reference value k is half the minimum shift (in σ units), so the CUSUM is tuned to shifts of about the minimum shift.
An alarm fires when either sum passes h. The change is placed at the first run after the sum was last zero, and the CUSUM then
restarts from there.

The CUSUM is **self-starting** (Hawkins 1987). The reference level μ0 of run t is the weighted mean of all runs since the
last restart, and testing starts after 5 warm-up runs. A first version used a fixed μ0 from the first 5 runs. At low
statistics its own error (σ/√5) was comparable to k, the null CUSUM then grew steadily, and the calibrated h rose to about 56.
The trace below has no restarts, so μ0 slowly absorbs the new level after the step and S+ levels off.

In [ ]:
frame = result["frames"][0]           # relative frame, mu
row, x = series_of("MVTX", 0, 5, 3, "rphi", frame.X)
s = frame.S[row]
plus, minus, level = rs.cusum_trace(x, s, frame.min_shift[row], baseline=5)
h = frame.thresholds["cusum_h"].iat[row]
offset = np.nanmedian(x[:5])
fig, (top, bottom) = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True, constrained_layout=True)
top.errorbar(np.arange(len(x)), (x - offset) / UM, yerr=s / UM, fmt="o", ms=4,
             color=METHOD_COLOR["cusum"], ecolor=MUTED, elinewidth=1, label="μ relative to layer")
top.plot(np.arange(len(x)), (level - offset) / UM, color=INK2, lw=1.5, label="reference level μ0")
top.legend(loc="lower right")
top.set_ylabel("µm (first runs subtracted)")
top.set_title("MVTX L0 stave 5 chip 3 r-φ (+10 µm from run 20)", loc="left", fontsize=10)
bottom.plot(plus, color=METHOD_COLOR["cusum"], label="S+")
bottom.plot(minus, color=INK2, ls=":", label="S−")
bottom.axhline(h, color=INK, lw=1, ls="--")
bottom.text(1, h * 1.08, f"h = {h:.1f}", fontsize=8, color=INK2)
bottom.set_yscale("symlog", linthresh=10)
bottom.set_ylabel("CUSUM (σ units)"); bottom.set_xlabel("run index"); bottom.legend(loc="upper left")
plt.show()

## 6. PELT

PELT (Killick, Fearnhead and Eckley 2012) finds the exact segmentation that minimises
$\sum_{\rm segments} \sum_i w_i (x_i - \bar x_w)^2 + \beta \cdot (\text{number of changes})$, with $w = 1/\sigma_{tot}^2$, so runs with
large errors count less. The penalty is $\beta = 2\ln(n N/E)$ (n runs, N series in the scan, E expected false changes per scan).
`rs.pelt` is the pruned algorithm. The scan solves the same recursion without pruning, vectorised over all series,
which gives identical change points (`tests/test_run_stability.py::test_pelt_scan_equals_pelt`).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=False, constrained_layout=True)
for ax, (name, key) in zip(axes, (("transient", examples["transient"]), ("drift", examples["drift"]))):
    row, x = series_of(*key, frame.X)
    s = frame.S[row]
    good = np.flatnonzero(np.isfinite(x))
    points = rs.pelt(x[good], s[good], frame.thresholds["pelt_beta"].iat[row])
    means, _ = rs.segment_means(x[good], s[good], points)
    bounds = [0, *points, len(good)]
    ax.errorbar(good, x[good] / UM, yerr=s[good] / UM, fmt="o", ms=4, color=MUTED,
                ecolor="#d6d5cf", elinewidth=1, label="μ relative to layer")
    for a, b, mean in zip(bounds[:-1], bounds[1:], means):
        ax.plot([good[a] - 0.5, good[b - 1] + 0.5], [mean / UM] * 2, color=METHOD_COLOR["pelt"])
    ax.set_title(f"{name}: PELT change points at runs {[int(good[p]) for p in points]}",
                 loc="left", fontsize=10)
    ax.set_xlabel("run index"); ax.set_ylabel("µm")
axes[0].legend(loc="upper left")
plt.show()

PELT turns a drift into a staircase. The classification therefore also fits a **ramp** (constant, then linear from
t1 to t2, then constant). A series is a `drift` if the ramp beats the step model by a BIC margin of 10, lasts at least 6 runs,
changes by more than the minimum shift and is significant. The margin covers the look-elsewhere advantage of searching all (t1, t2) pairs.

## 7. Bayesian online change-point detection

BOCPD (Adams and MacKay 2007) updates, after every run, the posterior of the **run length** r: the number of runs since
the last change. Each run uses a Normal likelihood with that run's own σ_tot, a Normal prior on each segment mean
(width 5 × minimum shift) and a constant hazard of 1/100 per run.
It works online: after run T it gives the probability q_T that a change happened within the last `lag + 1` = 4 runs.
That makes it suitable for monitoring during data taking. An alarm needs q above a threshold calibrated like h.

In [ ]:
row, x = series_of("MVTX", 0, 5, 3, "rphi", frame.X)
s = frame.S[row]
good = np.isfinite(x)
xg, sg = x[good], s[good]
mu0 = np.array([rs.weighted_mean(xg[:5], sg[:5])[0]])
q, r_map, posterior = rs.bocpd_batch(xg[None], sg[None], mu0, np.array([5 * frame.min_shift[row]]),
                                     1 / 100, 3, return_posterior=True)
threshold = frame.thresholds["bocpd_threshold"].iat[row]
fig, (top, bottom) = plt.subplots(2, 1, figsize=(11, 6), sharex=True, constrained_layout=True,
                                  gridspec_kw={"height_ratios": [2, 1]})
image = top.imshow(posterior.T, origin="lower", aspect="auto", cmap=SEQUENTIAL, vmin=0, vmax=1,
                   interpolation="nearest")
top.set_ylim(0, len(xg)); top.grid(False)
top.set_ylabel("run length r"); top.set_title("Run-length posterior P(r | data up to run)", loc="left", fontsize=10)
fig.colorbar(image, ax=top, label="probability")
shown = q[0].copy()
shown[:4] = np.nan          # runs 0..lag: no decision possible yet (q = 1)
bottom.plot(np.arange(len(xg)), shown, color=METHOD_COLOR["bocpd"], marker="^", ms=4)
bottom.axhline(threshold, color=INK, lw=1, ls="--")
bottom.text(5, threshold - 0.15, f"threshold = 1 − {1 - threshold:.0e}", fontsize=8, color=INK2)
bottom.set_ylabel("P(change in last 4 runs)"); bottom.set_xlabel("run index (valid runs)")
plt.show()

## 8. Thresholds and false alarms

The thresholds aim at about **one false change point per method in the whole scan** (E = 1). The scan covers about 5300 series, so the target per series is
p = E/N ≈ 2×10⁻⁴. CUSUM h, the BOCPD threshold and a Monte Carlo PELT penalty come from simulated Gaussian series.
Like the scan, each simulated series uses an error **estimated from the series itself** (fit error ⊕ MAD-based extra
scatter), so the thresholds include the uncertainty of that estimate. A first version that assumed known errors gave
about 25 times too many raw BOCPD alarms on the series with no changes. Below 10 expected exceedances, the quantile comes from an
exponential tail fitted above the 99% quantile. PELT uses the larger of 2 ln(nN/E) and the Monte Carlo value. Then every
change must also be confirmed by 2 of 3 methods, exceed the minimum shift (the `compare_runs` values) and have z > 5.

With estimated errors, BOCPD's Gaussian predictive is overconfident, and its threshold reaches the 1 − 10⁻⁹ cap in most
groups: it then fires only on strong evidence. A Student-t predictive (unknown variance) would be the standard improvement.

The calibration is only as good as the Gaussian-noise assumption. **On real data, measure the false-alarm rate on a
period known to be stable before trusting it.**

In [ ]:
pd.DataFrame(result["calibration"]).round(3)

In [ ]:
null_long, null_runs = rs.load_run_series(null_paths)
null_result = rs.scan(null_long, null_runs, rs.Config())
raw_detections = null_result["detections"].groupby(["frame", "quantity", "method"]).size()
print("Series with no injected change -> events:", len(null_result["events"]))
print("Raw per-method detections before consensus and minimum shift:")
print(raw_detections.to_string() if len(raw_detections) else "  none")

In [ ]:
# The null distribution behind h for MVTX r-phi (relative frame).
record = next(r for r in result["calibration"] if r["frame"] == "relative"
              and r["detector"] == "MVTX" and r["coord"] == "rphi")
rng = np.random.default_rng(1)
Z = rng.standard_normal((20000, len(runs)))
statistic = rs.cusum_null_statistic(Z, record["k"], 5,
                                    rs.simulated_errors(Z, max(record["fit_fraction"], 0.05)))
fig, ax = plt.subplots(figsize=(8, 3.4), constrained_layout=True)
ax.hist(statistic[statistic > 0], bins=60, color=METHOD_COLOR["cusum"], edgecolor=SURFACE, linewidth=0.5)
ax.axvline(record["cusum_h"], color=INK, ls="--", lw=1)
ax.text(record["cusum_h"] * 1.03, ax.get_ylim()[1] * 0.3, f"h = {record['cusum_h']:.2f}", fontsize=8)
ax.set_yscale("log"); ax.set_xlabel("largest CUSUM excursion in 60 runs (Gaussian noise)")
ax.set_ylabel("series"); ax.set_title(f"k = {record['k']:.2f}: {np.mean(statistic == 0):.0%} of series never leave 0",
                                      loc="left", fontsize=10)
plt.show()

## 9. Full scan versus the injected truth

`synthetic/check_run_stability.py` checks every injection:
- an expected class must appear at the right place and run (±2 runs; ±6 for the start and end of a drift);
- an injection that must not be reported (sub-threshold step and drift, low-statistics runs, missing ladder) must give nothing;
- everything else counts against a budget of 2 unexplained change points.

Per method, the table also shows whether each method alone fired on the injected sensors near the injected run,
before the consensus and the minimum shift.

In [ ]:
events = result["events"]
detections = result["detections"]
rows = []
for injection in gen.INJECTIONS:
    problem = truth.check_injection(injection, events)
    near = detections.loc[detections.apply(lambda r: truth.covers(injection, r), axis=1)] \
        if injection["kind"] not in ("lumi", "bad_run") else detections.iloc[0:0]
    start = injection.get("start", 0)
    near = near.loc[(near["run_index"] - start).abs() <= 2] if len(near) else near
    rows.append({"injection": injection["name"], "expected": injection["expected"] or "nothing",
                 **{method: bool((near["method"] == method).any()) if len(near) else False
                    for method in rs.METHODS},
                 "result": "ok" if problem is None else problem})
print("Unexplained change points:", len(truth.unexplained(events, gen.INJECTIONS)))
pd.DataFrame(rows)

The method columns are raw firings. A `False` for a "nothing" injection is the desired outcome. For the sub-threshold step the
methods may fire, since 3 µm is significant over 35 runs, but the minimum shift removes it. Drifts, outliers and bad runs are
found by the classification and the outlier pre-filter, not by a single change at `start`.

The events as written to `changepoints_*.csv` (grouped chip events hidden):

In [ ]:
paths = rs.write_outputs(result, runs, rs.Config(), OUT / "scan",
                         runs[["run", "csv", "sha256"]].to_dict(orient="records"))
columns = ["event_id", "class", "detector", "layer", "stave", "chip", "coord", "quantity",
           "frame", "run_index", "end_run_index", "delta_mm", "z", "methods", "n_members", "notes"]
events.loc[events["group_id"] == "", columns].round({"delta_mm": 4, "z": 1})

## 10. Detection efficiency versus step size

Single series with a step at run 30, of size f × minimum shift, at two noise levels: σ_tot = 1 µm (typical MVTX here) and
3 µm (low statistics). The thresholds are calibrated as in the full scan (p = 1/5300 per series). The coloured lines are each method
alone. The black line is the final result: 2 of 3 methods, larger than the minimum shift, z > 5 and classified as a step.

In [ ]:
def efficiency(sigma_um, fractions, per_size=150, n_runs=60, at=30, seed=5):
    rng = np.random.default_rng(seed)
    min_shift = 5 * UM
    rows = len(fractions) * per_size
    X = rng.normal(0, sigma_um * UM, (rows, n_runs))
    size = np.repeat(fractions, per_size) * min_shift
    X[:, at:] += size[:, None]
    keys_eff = pd.DataFrame({"detector": "MVTX", "layer": 0, "stave": np.arange(rows), "chip": 0,
                             "coord": "rphi"})
    cfg = rs.Config(expected_false=rows / 5300)   # same p per series as the full scan
    frame = rs.Frame("raw", "mu", keys_eff, X, np.full_like(X, sigma_um * UM), np.full(rows, min_shift))
    rs.calibrate_frames([frame], n_runs, cfg)
    found_events, found, _ = rs.scan_frame(frame, runs, cfg)
    hit = pd.DataFrame(found)
    hit = hit.loc[(hit["run_index"] - at).abs() <= 2] if len(hit) else hit
    table = {method: [hit.loc[hit["method"] == method, "row"].nunique() / per_size
                      if len(hit) else 0 for _ in [0]][0] for method in rs.METHODS}
    curves = {}
    for method in rs.METHODS:
        rows_hit = set(hit.loc[hit["method"] == method, "row"]) if len(hit) else set()
        curves[method] = [np.mean([r in rows_hit for r in range(i * per_size, (i + 1) * per_size)])
                          for i in range(len(fractions))]
    final = {e["row"] for e in found_events if e["class"] == "step" and abs(e["run_index"] - at) <= 2}
    curves["final"] = [np.mean([r in final for r in range(i * per_size, (i + 1) * per_size)])
                       for i in range(len(fractions))]
    return curves

fractions = np.array([0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0])
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True, constrained_layout=True)
for ax, sigma_um in zip(axes, (1.0, 3.0)):
    curves = efficiency(sigma_um, fractions)
    for method in rs.METHODS:
        ax.plot(fractions, curves[method], color=METHOD_COLOR[method], marker=METHOD_MARKER[method],
                ms=8, label=method)
    ax.plot(fractions, curves["final"], color=INK, marker="D", ms=7, ls="--", label="final (2 of 3, > min shift)")
    ax.axvline(1.0, color=MUTED, lw=1)
    ax.set_title(f"σ_tot = {sigma_um:g} µm, minimum shift 5 µm", loc="left", fontsize=10)
    ax.set_xlabel("step size / minimum shift")
for ax in axes:
    ax.legend(loc="lower right", fontsize=8)
axes[0].set_ylabel("detection probability")
plt.show()

How to read this:
- **Well below the minimum shift** (left of the gray line) the final result is quiet, even where single methods see the step.
- **Near the threshold** the estimated Δ is biased upward, because the change location is chosen where the difference looks
  largest. At σ_tot = 3 µm, about 10% of 3.75 µm steps (0.75×) are reported as above 5 µm. Treat changes within about 1.5×
  the minimum shift as uncertain in size.
- **BOCPD is the least sensitive at low statistics.** Its threshold is set high to cope with estimated errors (section 8).
- The numbers come from Gaussian noise. Real data with non-Gaussian run-to-run effects will do worse; the shape is the useful part.

## 11. Drifts, outlier runs and width changes

- **Drift.** CUSUM fires repeatedly and PELT makes a staircase. The ramp fit turns it into one `drift` event with a start, an end and
  a total change. A 4-run "drift" can't be told apart from a step plus run-to-run jitter, hence the 6-run minimum.
  A drift below the minimum shift in total (the INTT sensor, +10 µm over 60 runs) is not reported.
- **Outlier runs.** A point that disagrees with the runs on **both** sides by more than 6 σ_tot is an `outlier_run`
  and is left out before the change-point methods run. That is why a one-run spike is not reported as two steps.
- **Bad runs.** A run in which more than 20% of a detector's series are outliers is a `bad_run` for that detector, and is left out everywhere.
- **Width.** The same methods run on σ(run), with a minimum change of 20% of the reference width (as in `compare_runs`).

In [ ]:
frame_sigma = next(f for f in result["frames"] if f.quantity == "sigma")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), constrained_layout=True)
for ax, coord in zip(axes, ("rphi", "z")):
    row, x = series_of("MVTX", 0, 2, 5, coord, M["sigma"])
    ax.errorbar(np.arange(len(x)), x / UM, yerr=M["sigma_err"][row] / UM, fmt="o", ms=4,
                color=METHOD_COLOR["cusum"], ecolor=MUTED, elinewidth=1)
    ax.axvline(28, color=INK2, ls="--", lw=1)
    ax.axvline(47, color=MUTED, ls=":", lw=1)
    ax.text(47.3, np.nanmax(x) / UM * 0.98, "bad run", fontsize=8, color=INK2)
    ax.set_title(f"σ of MVTX L0 stave 2 chip 5 {coord} (×1.4 from run 28)", loc="left", fontsize=10)
    ax.set_xlabel("run index"); ax.set_ylabel("σ [µm]")
plt.show()
events.loc[events["class"].isin(["width_step", "drift", "outlier_run", "bad_run", "transient"]),
           ["event_id", "class", "detector", "layer", "stave", "chip", "coord", "quantity",
            "run_index", "end_run_index", "delta_mm", "methods"]]

## 12. Running it on real sPHENIX data

1. Make one ROOT file list per run (`list_run<N>.txt`), all reconstructed with **the same** alignment/geometry tag,
   field configuration, track selection and distortion corrections. Note in a `runs.csv` (columns `run`, `note`) the
   runs where anything changed: magnet ramps, accesses, cooling changes, MVTX half-barrel movements, new tags.
2. Fit every run:
   ```bash
   root -l -b -q 'macros/plot_fits_overlaid_stavechip.C+("list_run53001.txt:run53001,list_run53002.txt:run53002,...","",false)'
   ```
   This writes `residual_fits_run<N>.csv` per run (the macro takes any number of inputs; a long series can be
   fitted in batches).
3. Scan:
   ```bash
   python -m stdiag.run_stability residual_fits_run*.csv --run-info runs.csv
   ```
   The outputs go to `runs/stability_<first>_to_<last>/`: `changepoints_*.csv`, `stability_report_*.md`, `series_*.csv`,
   `detections_*.csv` and `stability_summary_*.json`.
4. **Before trusting any change point:** run the scan on a period you believe was stable. The number of events there is
   your real false-alarm rate. If it is far above about 1, raise `--z-min`, raise the minimum shifts (`--min-shift MVTX:rphi=0.008`)
   or look at the extra scatter in `series_*.csv` (`sigma_tot_mm` versus `fit_error_mm`).
5. To turn a residual change into a position change, confirm the sign convention of `residualX/Y` in MakeMilleFiles. Remember
   that biased residuals underestimate the displacement. For a cross-check, align the runs before and after the change
   separately with Millepede and compare the fitted parameters of the flagged sensors.

Not yet covered: stave rows (chip = −1) and VERTEX rows (beam-spot stability), and synthetic data that goes through the
ROOT fit macro.